# Get Coordinates

Geocodes housing addresses using the Google Maps API and saves lat/lon to `../final_project/geocoded.csv`.

> **Note:** Only run this if you need to regenerate `geocoded.csv`. The file is already committed to the repo. Requires a valid Google Maps Geocoding API key.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import googlemaps

## 1. Load and Filter Housing Data

Replicates just enough of the cleaning from `final_project.ipynb` to identify the rows with a recorded sale price — those are the rows we geocode.

In [2]:
df = pd.read_csv('../final_project/housing_data_2016_2017.csv', low_memory=False)
df = df.iloc[:, 28:]  # Drop first 28 irrelevant columns

# Parse sale_price to identify sold homes
df['sale_price'] = pd.to_numeric(
    df['sale_price'].astype(str).str.replace(r'[$,]', '', regex=True),
    errors='coerce'
)
df = df[df['sale_price'].notna()].reset_index(drop=True)

print(f'Rows to geocode: {len(df)}')
print(df['full_address_or_zip_code'].head())

Rows to geocode: 528
0                                   Flushing NY, 11355
1    30-11 Parsons Blvd,  Flushing NY, 11354 ( Sold...
2                  102-14 Lewis Ave,  Corona NY, 11368
3            144-48 Roosevelt Ave,  Flushing NY, 11354
4                245-27 76th Ave,  Bellerose NY, 11426
Name: full_address_or_zip_code, dtype: object


## 2. Initialize Google Maps Client

Set `GOOGLE_MAPS_API_KEY` as an environment variable, or enter it when prompted.

In [ ]:
api_key = os.environ.get('GOOGLE_MAPS_API_KEY', '')
if not api_key:
    api_key = input('Enter your Google Maps API key: ')

gmaps = googlemaps.Client(key=api_key)

## 3. Geocode Addresses

In [ ]:
lons, lats = [], []

for i, address in enumerate(df['full_address_or_zip_code']):
    try:
        result = gmaps.geocode(str(address))
        if result:
            loc = result[0]['geometry']['location']
            lons.append(loc['lng'])
            lats.append(loc['lat'])
        else:
            lons.append(np.nan)
            lats.append(np.nan)
    except Exception as e:
        print(f'Error at row {i}: {e}')
        lons.append(np.nan)
        lats.append(np.nan)

    if i % 50 == 0:
        print(f'Progress: {i}/{len(df)}')

    time.sleep(0.05)  # Respect API rate limits

# Store the address alongside the coordinates so the join in final_project.ipynb
geocoded = pd.DataFrame({
    'full_address_or_zip_code': df['full_address_or_zip_code'].values,
    'lon': lons,
    'lat': lats,
})
print(geocoded.describe())

## 4. Save

In [ ]:
geocoded.to_csv('../final_project/geocoded.csv', index=False)
print(f'Saved {len(geocoded)} rows to geocoded.csv')